# PreactResNet18 Backdoor Distillation

This notebook distills a student model from a backdoored PreactResNet18 teacher and checks whether the backdoor transfers to the student.

## 1. Paths and run selection


In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'defense').exists() and (candidate / 'record').exists() and (candidate / 'notebooks_finetuning').exists():
            return candidate
    raise FileNotFoundError('Could not locate the cloned backdoor_finetuning repo. Start Jupyter from the repo or its notebooks_finetuning folder.')


BACKDOOR_REPO = find_repo_root()
NOTEBOOK_DIR = BACKDOOR_REPO / 'notebooks_finetuning'
DISTILLATION_DIR = NOTEBOOK_DIR / 'distillation'
ZIP_SEARCH_DIRS = [DISTILLATION_DIR, NOTEBOOK_DIR]
DISTILLATION_DIR.mkdir(parents=True, exist_ok=True)

available_zips = sorted({p.name for folder in ZIP_SEARCH_DIRS if folder.exists() for p in folder.glob('cifar10_preactresnet18_*.zip')})
print('Available PreactResNet18 runs:')
for name in available_zips:
    print('  -', name)

# Pick the run to evaluate here.
ZIP_NAME = 'cifar10_preactresnet18_sig_0_01.zip'


def resolve_zip_path(zip_name):
    zip_path = Path(zip_name).expanduser()
    if zip_path.is_absolute():
        return zip_path
    for folder in ZIP_SEARCH_DIRS:
        candidate = folder / zip_path
        if candidate.exists():
            return candidate
    return DISTILLATION_DIR / zip_path


ZIP_PATH = resolve_zip_path(ZIP_NAME)
RUN_NAME = ZIP_PATH.stem
EXTRACT_ROOT = DISTILLATION_DIR / 'extracted'
OUTPUT_ROOT = DISTILLATION_DIR / 'outputs'
DATA_ROOT = DISTILLATION_DIR / 'data'

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Could not find {ZIP_PATH}. Choose one of the listed zip files or place it under {DISTILLATION_DIR}.')
if not BACKDOOR_REPO.exists():
    raise FileNotFoundError(f'Could not find the BackdoorBench repo at {BACKDOOR_REPO}')

print('\nSelected run:', RUN_NAME)
print('Backdoor repo:', BACKDOOR_REPO)


## 2. Imports, device, and experiment settings


In [ ]:
import importlib.util
import io
import json
import os
import random
import shutil
import sys
import time
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

missing = []
for module_name, package_name in [
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('pandas', 'pandas'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
    ('PIL', 'pillow'),
]:
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)

if missing:
    raise ImportError(
        'Missing packages: ' + ', '.join(missing) +
        '. Run the optional install cell above, then restart this kernel.'
    )

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm

sys.path.insert(0, str(BACKDOOR_REPO))
try:
    from utils.aggregate_block.model_trainer_generate import generate_cls_model
except Exception as exc:
    print('Falling back to direct PreActResNet18 import because generate_cls_model failed:')
    print(repr(exc))
    from models.preact_resnet import PreActResNet18

    def generate_cls_model(model_name, num_classes=10, **kwargs):
        if model_name.lower() != 'preactresnet18':
            raise ValueError(f'Fallback loader only supports preactresnet18, got {model_name}')
        return PreActResNet18(num_classes=num_classes)


def pick_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = pick_device()
SEED = 0


STUDENT_MODEL = 'preactresnet18'
EPOCHS = 10
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
NUM_WORKERS = 0
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
TEMPERATURE = 4.0
ALPHA = 0.9
CLEAN_TRAIN_LIMIT = None       
DOWNLOAD_CIFAR10 = True       
INIT_STUDENT_FROM_TEACHER = False
DISTILL_TRAIN_SOURCE = 'clean_cifar10'  


TARGET_LABEL_OVERRIDE = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == 'mps' and hasattr(torch, 'mps') and hasattr(torch.mps, 'manual_seed'):
    torch.mps.manual_seed(SEED)

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'mps':
    print('Using Apple Metal Performance Shaders. If you hit an unsupported-op error, set DEVICE = torch.device("cpu").')

## 3. Extract the selected zip

In [ ]:
def extract_run(zip_path, extract_root, force=False):
    run_dir = extract_root / zip_path.stem
    marker = run_dir / '.extract_complete'

    if force and run_dir.exists():
        shutil.rmtree(run_dir)

    if marker.exists():
        print('Already extracted:', run_dir)
        return run_dir

    run_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting', zip_path.name, 'to', run_dir)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(run_dir)
    marker.write_text(datetime.now().isoformat())
    return run_dir

run_dir = extract_run(ZIP_PATH, EXTRACT_ROOT, force=False)
print('Run files:', sorted(p.name for p in run_dir.iterdir()))

for required in ['attack_result.pt', 'bd_train_dataset', 'bd_test_dataset']:
    if not (run_dir / required).exists():
        raise FileNotFoundError(f'Missing {required} in {run_dir}')

## 4. Dataset helpers


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.247, 0.243, 0.261)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomCrop((32, 32), padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

class IntFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        if not self.root.exists():
            raise FileNotFoundError(self.root)

        class_dirs = [p for p in self.root.iterdir() if p.is_dir()]
        class_dirs = sorted(class_dirs, key=lambda p: int(p.name) if p.name.isdigit() else p.name)
        for class_dir in class_dirs:
            try:
                label = int(class_dir.name)
            except ValueError as exc:
                raise ValueError(f'Expected numeric class folder, got {class_dir.name}') from exc
            for image_path in sorted(class_dir.rglob('*')):
                if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    self.samples.append((image_path, label))

        if not self.samples:
            raise ValueError(f'No images found under {self.root}')
        self.labels = [label for _, label in self.samples]
        self.classes = sorted(set(self.labels))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def maybe_limit_dataset(dataset, limit, seed=0):
    if limit is None or limit >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)

bd_train_dataset = IntFolderDataset(run_dir / 'bd_train_dataset', transform=train_transform)
bd_test_dataset = IntFolderDataset(run_dir / 'bd_test_dataset', transform=test_transform)

if DISTILL_TRAIN_SOURCE == 'clean_cifar10':
    train_dataset = CIFAR10(root=str(DATA_ROOT), train=True, download=DOWNLOAD_CIFAR10, transform=train_transform)
    train_dataset = maybe_limit_dataset(train_dataset, CLEAN_TRAIN_LIMIT, seed=SEED)
elif DISTILL_TRAIN_SOURCE == 'bd_train_folder':
    train_dataset = maybe_limit_dataset(bd_train_dataset, CLEAN_TRAIN_LIMIT, seed=SEED)
else:
    raise ValueError("DISTILL_TRAIN_SOURCE must be 'clean_cifar10' or 'bd_train_folder'")

clean_test_dataset = CIFAR10(root=str(DATA_ROOT), train=False, download=DOWNLOAD_CIFAR10, transform=test_transform)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
clean_test_loader = DataLoader(clean_test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, **loader_kwargs)
bd_test_loader = DataLoader(bd_test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, **loader_kwargs)

print('Distillation train source:', DISTILL_TRAIN_SOURCE)
print('Train samples:', len(train_dataset))
print('Clean test samples:', len(clean_test_dataset))
print('BD train samples in zip:', len(bd_train_dataset), Counter(bd_train_dataset.labels))
print('BD test samples in zip:', len(bd_test_dataset), Counter(bd_test_dataset.labels))

## 5. Load teacher and create student


In [ ]:
def torch_load_compat(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def strip_module_prefix(state_dict):
    return {
        key[7:] if key.startswith('module.') else key: value
        for key, value in state_dict.items()
    }


def looks_like_state_dict(candidate):
    if not isinstance(candidate, dict) or not candidate:
        return False

    values = list(candidate.values())
    tensor_count = sum(torch.is_tensor(value) for value in values)
    if tensor_count == 0:
        return False
    if tensor_count == len(values):
        return True

    tensor_fraction = tensor_count / len(values)
    has_parameter_keys = any(
        isinstance(key, str)
        and (
            '.' in key
            or key.endswith(('weight', 'bias', 'running_mean', 'running_var', 'num_batches_tracked'))
        )
        for key, value in candidate.items()
        if torch.is_tensor(value)
    )
    return tensor_fraction >= 0.5 and has_parameter_keys


def extract_state_dict(obj):
    if isinstance(obj, dict):
        if looks_like_state_dict(obj):
            return obj
        for key in ('model', 'model_state_dict', 'state_dict', 'net', 'network'):
            if key in obj:
                return extract_state_dict(obj[key])
    raise ValueError('Could not find a model state_dict in the checkpoint.')


def load_state_dict_flexible(model, state_dict):
    state_dict = strip_module_prefix(state_dict)
    try:
        model.load_state_dict(state_dict, strict=True)
        return 'strict'
    except RuntimeError as strict_error:
        model_state = model.state_dict()
        model_keys = list(model_state.keys())
        state_keys = list(state_dict.keys())
        if len(model_keys) == len(state_keys):
            remapped = {}
            compatible = True
            for model_key, state_key in zip(model_keys, state_keys):
                if tuple(model_state[model_key].shape) != tuple(state_dict[state_key].shape):
                    compatible = False
                    break
                remapped[model_key] = state_dict[state_key]
            if compatible:
                print(
                    'WARNING: strict state-dict loading failed; falling back to positional key remapping. '
                    'Verify that the checkpoint architecture matches the model.'
                )
                model.load_state_dict(remapped, strict=True)
                return 'remapped_by_order'
        raise strict_error


def nested_get(obj, name):
    if isinstance(obj, dict):
        return obj.get(name)
    return getattr(obj, name, None)


def coerce_int(value):
    if value is None:
        return None
    if hasattr(value, 'item'):
        value = value.item()
    if isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def infer_target_label(raw_attack, bd_dataset, num_classes):
    if TARGET_LABEL_OVERRIDE is not None:
        return int(TARGET_LABEL_OVERRIDE), 'override'

    for key in ('target_label', 'attack_target', 'target_class', 'poison_label', 'target'):
        value = coerce_int(raw_attack.get(key)) if isinstance(raw_attack, dict) else None
        if value is not None:
            return value, f'attack_result[{key!r}]'

    args_obj = raw_attack.get('args') if isinstance(raw_attack, dict) else None
    for key in ('target_label', 'attack_target', 'target_class', 'poison_label', 'target'):
        value = coerce_int(nested_get(args_obj, key))
        if value is not None:
            return value, f'attack_result.args.{key}'

    missing = sorted(set(range(num_classes)) - set(bd_dataset.classes))
    if len(missing) == 1:
        return missing[0], 'missing class folder in bd_test_dataset'

    return 0, 'fallback default'

raw_attack = torch_load_compat(run_dir / 'attack_result.pt', map_location='cpu')
print('attack_result keys:', sorted(raw_attack.keys()) if isinstance(raw_attack, dict) else type(raw_attack))

teacher_model_name = raw_attack.get('model_name', 'preactresnet18') if isinstance(raw_attack, dict) else 'preactresnet18'
teacher_model_name = str(teacher_model_name).lower()
num_classes = int(raw_attack.get('num_classes', 10)) if isinstance(raw_attack, dict) else 10
target_label, target_source = infer_target_label(raw_attack, bd_test_dataset, num_classes)

teacher = generate_cls_model(teacher_model_name, num_classes=num_classes)
teacher_state = extract_state_dict(raw_attack)
teacher_load_mode = load_state_dict_flexible(teacher, teacher_state)
teacher = teacher.to(DEVICE).eval()

student = generate_cls_model(STUDENT_MODEL, num_classes=num_classes)
student_init = 'random'
if INIT_STUDENT_FROM_TEACHER:
    student_load_mode = load_state_dict_flexible(student, teacher_state)
    student_init = f'teacher_weights/{student_load_mode}'
student = student.to(DEVICE)

print('Teacher model:', teacher_model_name)
print('Teacher load mode:', teacher_load_mode)
print('Student model:', STUDENT_MODEL)
print('Student init:', student_init)
print('Num classes:', num_classes)
print('Target label:', target_label, f'({target_source})')

## 6. Evaluate the teacher before distillation


In [ ]:
def to_device(batch):
    x, y = batch[0], batch[1]
    non_blocking = DEVICE.type == 'cuda'
    return x.to(DEVICE, non_blocking=non_blocking), y.to(DEVICE, non_blocking=non_blocking)

@torch.no_grad()
def evaluate_accuracy(model, loader, forced_target=None):
    model.eval()
    correct = 0
    total = 0
    for batch in tqdm(loader, leave=False):
        x, y = to_device(batch)
        logits = model(x)
        pred = logits.argmax(dim=1)
        if forced_target is None:
            correct += (pred == y).sum().item()
        else:
            target = torch.full_like(y, int(forced_target))
            correct += (pred == target).sum().item()
        total += y.numel()
    return correct / max(total, 1)

@torch.no_grad()
def prediction_histogram(model, loader, num_classes):
    model.eval()
    counts = torch.zeros(num_classes, dtype=torch.long)
    for batch in tqdm(loader, leave=False):
        x, _ = to_device(batch)
        pred = model(x).argmax(dim=1).detach().cpu()
        counts += torch.bincount(pred, minlength=num_classes)
    return {int(i): int(v) for i, v in enumerate(counts.tolist())}


def evaluate_backdoor_suite(model, include_histogram=False):
    clean_acc = evaluate_accuracy(model, clean_test_loader)
    asr = evaluate_accuracy(model, bd_test_loader, forced_target=target_label)
    ra = evaluate_accuracy(model, bd_test_loader)
    metrics = {
        'clean_acc': clean_acc,
        'asr': asr,
        'ra': ra,
    }
    if include_histogram:
        metrics['bd_pred_hist'] = prediction_histogram(model, bd_test_loader, num_classes)
    return metrics

teacher_metrics = evaluate_backdoor_suite(teacher, include_histogram=True)
pd.DataFrame([{
    'model': 'teacher',
    'clean_acc': teacher_metrics['clean_acc'],
    'asr': teacher_metrics['asr'],
    'ra': teacher_metrics['ra'],
}])

## 7. Distill the student

The loss is a weighted sum of soft-label knowledge distillation and ordinary cross entropy on the clean labels:

loss = ALPHA * KD(student, teacher, TEMPERATURE) + (1 - ALPHA) * CE(student, labels)

- ALPHA = 1.0: pure distillation from teacher soft labels.
- ALPHA = 0.0: supervised clean fine-tuning baseline with no teacher signal.
- INIT_STUDENT_FROM_TEACHER = True: fine-tune a teacher-initialized student instead of training from scratch.

In [ ]:
optimizer = torch.optim.SGD(
    student.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
print(
    f"Teacher | clean_acc {teacher_metrics['clean_acc']:.4f} | "
    f"ASR {teacher_metrics['asr']:.4f} | RA {teacher_metrics['ra']:.4f}"
)

for epoch in range(1, EPOCHS + 1):
    student.train()
    running_loss = 0.0
    running_kd = 0.0
    running_ce = 0.0
    total = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for batch in progress:
        x, y = to_device(batch)

        with torch.no_grad():
            teacher_logits = teacher(x)

        student_logits = student(x)
        kd_loss = F.kl_div(
            F.log_softmax(student_logits / TEMPERATURE, dim=1),
            F.softmax(teacher_logits / TEMPERATURE, dim=1),
            reduction='batchmean',
        ) * (TEMPERATURE * TEMPERATURE)
        ce_loss = F.cross_entropy(student_logits, y)
        loss = ALPHA * kd_loss + (1.0 - ALPHA) * ce_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_size = y.numel()
        running_loss += loss.item() * batch_size
        running_kd += kd_loss.item() * batch_size
        running_ce += ce_loss.item() * batch_size
        total += batch_size
        progress.set_postfix(loss=running_loss / max(total, 1))

    scheduler.step()

    student_metrics = evaluate_backdoor_suite(student)
    row = {
        'epoch': epoch,
        'loss': running_loss / max(total, 1),
        'kd_loss': running_kd / max(total, 1),
        'ce_loss': running_ce / max(total, 1),
        'clean_acc': student_metrics['clean_acc'],
        'asr': student_metrics['asr'],
        'ra': student_metrics['ra'],
        'lr': scheduler.get_last_lr()[0],
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d} | loss {row['loss']:.4f} | "
        f"clean_acc {row['clean_acc']:.4f} | ASR {row['asr']:.4f} | RA {row['ra']:.4f}"
    )

history_df = pd.DataFrame(history)
history_df

## 8. Plot and interpret


In [ ]:
if history:
    ax = history_df.plot(x='epoch', y=['clean_acc', 'asr', 'ra'], marker='o', figsize=(8, 4))
    ax.axhline(1.0 / num_classes, color='gray', linestyle='--', linewidth=1, label='chance ASR')
    ax.set_ylim(0, 1)
    ax.set_ylabel('rate')
    ax.set_title(RUN_NAME)
    ax.legend()
    plt.show()

student_final_metrics = evaluate_backdoor_suite(student, include_histogram=True)
final_student = student_final_metrics
summary_df = pd.DataFrame([
    {
        'model': 'teacher',
        'clean_acc': teacher_metrics['clean_acc'],
        'asr': teacher_metrics['asr'],
        'ra': teacher_metrics['ra'],
    },
    {
        'model': 'student_final',
        'clean_acc': final_student['clean_acc'],
        'asr': final_student['asr'],
        'ra': final_student['ra'],
    },
])
display(summary_df)

chance_asr = 1.0 / num_classes
student_asr = float(final_student['asr'])
if student_asr > 0.5:
    print(f'Interpretation: student ASR is high ({student_asr:.3f}), so the backdoor likely survived distillation.')
elif student_asr <= chance_asr * 2:
    print(f'Interpretation: student ASR is near chance ({student_asr:.3f} vs {chance_asr:.3f}), so distillation likely weakened the backdoor.')
else:
    print(f'Interpretation: student ASR is reduced but above chance ({student_asr:.3f} vs {chance_asr:.3f}); compare against more epochs and ALPHA settings.')

print('Teacher triggered prediction histogram:', teacher_metrics['bd_pred_hist'])
print('Student triggered prediction histogram:', prediction_histogram(student, bd_test_loader, num_classes))

## 9. Save student and metrics

In [ ]:
output_dir = OUTPUT_ROOT / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

student_path = output_dir / 'distilled_student.pt'
history_path = output_dir / 'distillation_history.csv'
metrics_path = output_dir / 'distillation_metrics.json'
summary_path = output_dir / 'summary.csv'

student_cpu_state = {k: v.detach().cpu() for k, v in student.state_dict().items()}
torch.save(student_cpu_state, student_path)
history_df.to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)

metrics = {
    'run_name': RUN_NAME,
    'zip_name': ZIP_NAME,
    'teacher': {
        'model_name': teacher_model_name,
        'clean_acc': teacher_metrics['clean_acc'],
        'asr': teacher_metrics['asr'],
        'ra': teacher_metrics['ra'],
        'bd_pred_hist': teacher_metrics['bd_pred_hist'],
    },
    'student_final': {
        'model_name': STUDENT_MODEL,
        'clean_acc': float(final_student['clean_acc']),
        'asr': float(final_student['asr']),
        'ra': float(final_student['ra']),
        'bd_pred_hist': final_student.get('bd_pred_hist'),
    },
    'settings': {
        'device': str(DEVICE),
        'seed': SEED,
        'target_label': target_label,
        'target_source': target_source,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'momentum': MOMENTUM,
        'weight_decay': WEIGHT_DECAY,
        'temperature': TEMPERATURE,
        'alpha': ALPHA,
        'clean_train_limit': CLEAN_TRAIN_LIMIT,
        'init_student_from_teacher': INIT_STUDENT_FROM_TEACHER,
        'distill_train_source': DISTILL_TRAIN_SOURCE,
    },
}
metrics_path.write_text(json.dumps(metrics, indent=2))

print('Saved student:', student_path)
print('Saved history:', history_path)
print('Saved metrics:', metrics_path)
print('Saved summary:', summary_path)

In [ ]:
# Save distillation plots
plot_dir = output_dir / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

# 1. Training/evaluation curves
if history:
    fig, ax = plt.subplots(figsize=(8, 4))
    history_df.plot(
        x="epoch",
        y=["clean_acc", "asr", "ra"],
        marker="o",
        ax=ax,
    )
    ax.axhline(
        1.0 / num_classes,
        color="gray",
        linestyle="--",
        linewidth=1,
        label="chance ASR",
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("rate")
    ax.set_title(RUN_NAME)
    ax.legend()
    fig.tight_layout()

    curves_path = plot_dir / "distillation_curves.png"
    fig.savefig(curves_path, dpi=200, bbox_inches="tight")
    plt.show()

# 2. Teacher vs student bar chart
fig, ax = plt.subplots(figsize=(7, 4))
summary_df.set_index("model")[["clean_acc", "asr", "ra"]].plot(
    kind="bar",
    ax=ax,
)
ax.axhline(
    1.0 / num_classes,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="chance ASR",
)
ax.set_ylim(0, 1)
ax.set_ylabel("rate")
ax.set_title(f"Teacher vs Student: {RUN_NAME}")
ax.legend()
fig.tight_layout()

bar_path = plot_dir / "teacher_student_summary.png"
fig.savefig(bar_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plots to:", plot_dir)